# Notebook 3: Reform RDF triples into a property graph

Import the Python library dependencies.

In [1]:
import json
import pathlib
import re
import shutil
import sys

from icecream import ic
import maplib
import polars as pl
import watermark

import kuzu
#import ryu

Run a "watermark" to show which library versions are used in this notebook's runtime environment.

In [2]:
%load_ext watermark
%watermark
%watermark --iversions

Last updated: 2025-11-12T18:10:16.240263-08:00

Python implementation: CPython
Python version       : 3.13.8
IPython version      : 9.1.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 24.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

json     : 2.0.9
sys      : 3.13.8 (main, Oct  7 2025, 12:01:51) [Clang 17.0.0 (clang-1700.0.13.3)]
watermark: 2.5.0
kuzu     : 0.9.0
polars   : 1.29.0
re       : 2.2.1
maplib   : 0.17.12



## SPARQL queries in Maplib

The `thesaurus.ttl` file provides a _semantic graph_, which we need to reform as a _property graph_.
We'll run SPARQL queries in [`Maplib`](https://github.com/DataTreehouse/maplib), where the result sets from these queries become [`Polars` dataframes](https://docs.pola.rs/api/python/stable/reference/dataframe/index.html), which can then be loaded as tables in [`RyuGraph`](https://ryugraph.io/).

First we load the RDF triples into a `Model` and define RDF [_prefix names_](https://www.w3.org/TR/sparql11-query/#prefNames) used to make SPARQL queries more compact.

In [3]:
rdf_model: maplib.Model = maplib.Model()
rdf_model.read(pathlib.Path("thesaurus.ttl"))

PREFIX_NAMES: str = """
PREFIX dc:   <http://purl.org/dc/elements/1.1/>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX sz:   <https://github.com/senzing-garage/sz-semantics/wiki/ns#>
"""

Define an accessor function to streamline how to run SPARQL queries in Maplib.

**Note: this is specific to this use case**, since Maplib does not yet appear to have N3 serialization support?

In [4]:
def run_sparql (
    query: str,
    ) -> pl.DataFrame:
    pat = re.compile(r"(\\?\w+)")
    rs: dict = json.loads(rdf_model.query(PREFIX_NAMES + query, return_json = True))
    rows: list = []

    for line in query.split("\n"):
        if line.strip().startswith("SELECT"):
            cols: list[ str] = re.findall(pat, line.replace("SELECT ", ""))

    for row in rs["results"]["bindings"]:
        row_dict: dict = {}

        for name, attr in row.items():
            row_dict[name] = attr["value"]

        rows.append(row_dict)

    return pl.from_dicts(rows).select(cols)

Build a dataframe for the taxonomy nodes.

In [5]:
df_taxo_nodes: pl.DataFrame = run_sparql("""
SELECT ?id ?descrip
WHERE {
  ?id a skos:Concept ;
    skos:definition ?descrip .
}""")

df_taxo_nodes.replace_column(
    0,
    df_taxo_nodes["id"].str.replace("https://github.com/senzing-garage/sz-semantics/wiki/ns#", "sz:"),
)

df_taxo_nodes = df_taxo_nodes.with_columns(pl.lit("skos:Concept").alias("class"))

df_taxo_nodes.head(3)

id,descrip,class
str,str,str
"""sz:DataRecord""","""Data Record is a composite dat…","""skos:Concept"""
"""sz:Entity""","""Entity is anything that can be…","""skos:Concept"""
"""sz:Organization""","""Organization is a social entit…","""skos:Concept"""


Build a dataframe for the `sz:Person` nodes from ER.

In [6]:
df_er_person_nodes: pl.DataFrame = run_sparql("""
SELECT ?id ?descrip
WHERE {
  ?id a sz:Person ;
    skos:prefLabel ?descrip .
}""")

df_er_person_nodes.replace_column(
    0,
    df_er_person_nodes["id"].str.replace("https://github.com/senzing-garage/sz-semantics/wiki/ns#", "sz:"),
)

df_er_person_nodes = df_er_person_nodes.with_columns(pl.lit("sz:Person").alias("class"))

df_er_person_nodes.head(3)

id,descrip,class
str,str,str
"""sz:1""","""Abassin Badshah""","""sz:Person"""
"""sz:10""","""Nicholas Thomas Wright""","""sz:Person"""
"""sz:102""","""Sarjit Kaur""","""sz:Person"""


Build a dataframe for the `sz:Organization` nodes from ER.

In [7]:
df_er_organ_nodes: pl.DataFrame = run_sparql("""
SELECT ?id ?descrip
WHERE {
  ?id a sz:Organization ;
    skos:prefLabel ?descrip .
}""")

df_er_organ_nodes.replace_column(
    0,
    df_er_organ_nodes["id"].str.replace("https://github.com/senzing-garage/sz-semantics/wiki/ns#", "sz:"),
)

df_er_organ_nodes = df_er_organ_nodes.with_columns(pl.lit("sz:Organization").alias("class"))

df_er_organ_nodes.head(3)

id,descrip,class
str,str,str
"""sz:100""","""GREENLIGHT GROUP LIMITED""","""sz:Organization"""
"""sz:101""","""TMF CORPORATE SERVICES LIMITED""","""sz:Organization"""
"""sz:104""","""Victor Nyland Invest ApS""","""sz:Organization"""


Build a dataframe for the `sz:DataRecord` nodes from ER.

In [8]:
df_er_data_nodes: pl.DataFrame = run_sparql("""
SELECT ?id ?data_src ?rec_key
WHERE {
  ?id a sz:DataRecord ;
    prov:wasQuotedFrom ?data_src ;
    dc:identifier ?rec_key .
}""")

df_er_data_nodes.replace_column(
    0,
    df_er_data_nodes["id"].str.replace("https://github.com/senzing-garage/sz-semantics/wiki/ns#", "sz:"),
)

df_er_data_nodes.replace_column(
    1,
    df_er_data_nodes["data_src"].str.replace("https://github.com/senzing-garage/sz-semantics/wiki/ns#", "sz:"),
)

df_er_data_nodes = df_er_data_nodes.with_columns(
    pl.concat_str(
        [
            pl.col("data_src"),
            pl.col("rec_key"),
        ],
        separator = ":",
    ).alias("descrip"),
)

df_er_data_nodes.drop_in_place("data_src")
df_er_data_nodes.drop_in_place("rec_key")
df_er_data_nodes = df_er_data_nodes.with_columns(pl.lit("sz:DataRecord").alias("class"))

df_er_data_nodes.head(3)

id,descrip,class
str,str,str
"""sz:ds_open-ownership_100945215…","""sz:ds_open-ownership:100945215…","""sz:DataRecord"""
"""sz:ds_open-ownership_101656327…","""sz:ds_open-ownership:101656327…","""sz:DataRecord"""
"""sz:ds_open-ownership_102644597…","""sz:ds_open-ownership:102644597…","""sz:DataRecord"""


Build a dataframe for the relations among entities and source data records.

In [9]:
df_er_rel_nodes: pl.DataFrame = run_sparql("""
SELECT ?ent ?rel_ent ?sem_rel ?why ?evidence
WHERE {
  ?bl rdf:predicate ?sem_rel ;
    rdf:subject ?ent ;
    rdf:object ?rel_ent ;
    sz:match_level ?why ;
    sz:match_key ?evidence .
}""")

df_er_rel_nodes.replace_column(
    0,
    df_er_rel_nodes["ent"].str.replace("https://github.com/senzing-garage/sz-semantics/wiki/ns#", "sz:"),
)

df_er_rel_nodes.replace_column(
    1,
    df_er_rel_nodes["rel_ent"].str.replace("https://github.com/senzing-garage/sz-semantics/wiki/ns#", "sz:"),
)

df_er_rel_nodes.replace_column(
    2,
    df_er_rel_nodes["sem_rel"].str.replace("http://www.w3.org/2004/02/skos/core#", "skos:"),
)

df_er_rel_nodes.head(3)

ent,rel_ent,sem_rel,why,evidence
str,str,str,str,str
"""sz:111""","""sz:80""","""skos:related""","""DISCLOSED""","""+OOR(:SHAREHOLDING 25% 50%)"""
"""sz:124""","""sz:160""","""skos:related""","""POSSIBLY_RELATED""","""+ADDRESS+REGISTRATION_COUNTRY-…"
"""sz:132""","""sz:245""","""skos:related""","""DISCLOSED""","""+OOR(:OTHER_INFLUENCE_OR_CONTR…"


## Graph database tables

This part of the tutorial was orginally written by [Prashanth Rao](https://github.com/prrao87) using `KùzuDB`, which is an embedded, open source graph database that supports the [Cypher](https://opencypher.org/) query language. It uses a _structured property graph_ model, which is similar to the _labeled property graph_ model you may be familiar with from other systems. The only difference being that KùzuDB requires strict data types for properties in the schema.

**The next step will delete any previous tables.**
The following steps then create the graph schema -- i.e., the node and relationship tables -- and populate data into them.

In [10]:
DB_PATH: str = "./db"
shutil.rmtree(DB_PATH, ignore_errors = True)

In [11]:
if "kuzu" in sys.modules:
    db: kuzu.Database = kuzu.Database(DB_PATH)
    conn: kuzu.Connection = kuzu.Connection(db)
else:
    db: ryu.Database = ryu.Database(DB_PATH)
    conn: ryu.Connection = ryu.Connection(db)

In [12]:
conn.execute("DROP TABLE IF EXISTS Related");
conn.execute("DROP TABLE IF EXISTS Entity");

In [13]:
conn.execute("CREATE NODE TABLE IF NOT EXISTS Entity (id STRING PRIMARY KEY, descrip STRING, class STRING)");
conn.execute("COPY Entity FROM (LOAD FROM df_taxo_nodes RETURN id, descrip, class)");
conn.execute("COPY Entity FROM (LOAD FROM df_er_person_nodes RETURN id, descrip, class)");
conn.execute("COPY Entity FROM (LOAD FROM df_er_organ_nodes RETURN id, descrip, class)");
conn.execute("COPY Entity FROM (LOAD FROM df_er_data_nodes RETURN id, descrip, class)");

In [14]:
conn.execute("CREATE REL TABLE IF NOT EXISTS Related (FROM Entity TO Entity, sem_rel STRING, why STRING, evidence STRING)");
conn.execute("COPY Related FROM df_er_rel_nodes");

In [15]:
conn.execute("DROP TABLE IF EXISTS OpenSanctions");
conn.execute("DROP TABLE IF EXISTS OpenOwnership");

In [16]:
df_os: pl.DataFrame = pl.read_csv("os_data.csv", separator = ",")

In [17]:
conn.execute("CREATE NODE TABLE IF NOT EXISTS OpenSanctions (id STRING PRIMARY KEY, descrip STRING, class STRING, addr STRING, url STRING)");
conn.execute("COPY OpenSanctions FROM df_os");

In [18]:
df_oo: pl.DataFrame = pl.read_csv("oo_data.csv", separator = ",")

In [19]:
conn.execute("CREATE NODE TABLE IF NOT EXISTS OpenOwnership (id STRING PRIMARY KEY, descrip STRING, class STRING, addr STRING, country STRING)");
conn.execute("COPY OpenOwnership FROM df_oo");

In [20]:
df_risk: pl.DataFrame = pl.read_csv("os_risk.csv", separator = ",")

In [21]:
conn.execute("DROP TABLE IF EXISTS Risk");
conn.execute("DROP TABLE IF EXISTS Role");

In [22]:
conn.execute("CREATE NODE TABLE IF NOT EXISTS Risk (topic STRING PRIMARY KEY)");
conn.execute("COPY Risk FROM (LOAD FROM df_risk RETURN DISTINCT topic)");

In [23]:
df_ubos: pl.DataFrame = pl.read_csv("oo_ubos.csv", separator = ",")

In [24]:
conn.execute("CREATE REL TABLE IF NOT EXISTS Role (FROM OpenOwnership TO OpenOwnership, role STRING, date DATE)");
conn.execute("COPY Role FROM df_ubos");

In [25]:
conn.execute("DROP TABLE IF EXISTS HasRisk");
conn.execute("CREATE REL TABLE IF NOT EXISTS HasRisk (FROM OpenSanctions TO Risk)");
conn.execute("COPY HasRisk FROM df_risk");

In [26]:
df_er_rel_nodes.head(3)

ent,rel_ent,sem_rel,why,evidence
str,str,str,str,str
"""sz:111""","""sz:80""","""skos:related""","""DISCLOSED""","""+OOR(:SHAREHOLDING 25% 50%)"""
"""sz:124""","""sz:160""","""skos:related""","""POSSIBLY_RELATED""","""+ADDRESS+REGISTRATION_COUNTRY-…"
"""sz:132""","""sz:245""","""skos:related""","""DISCLOSED""","""+OOR(:OTHER_INFLUENCE_OR_CONTR…"


In [27]:
df_er_os: pl.DataFrame = df_er_rel_nodes.filter(
    pl.col("rel_ent").str.starts_with("sz:ds_open-sanctions")
)

df_er_os.head(3)

ent,rel_ent,sem_rel,why,evidence
str,str,str,str,str
"""sz:1""","""sz:ds_open-sanctions_NK-25vyVF…","""skos:exactMatch""","""INITIAL""","""INITIAL"""
"""sz:11""","""sz:ds_open-sanctions_Q4342439""","""skos:exactMatch""","""INITIAL""","""INITIAL"""
"""sz:9""","""sz:ds_open-sanctions_NK-SKAADA…","""skos:exactMatch""","""INITIAL""","""INITIAL"""


In [28]:
df_er_oo: pl.DataFrame = df_er_rel_nodes.filter(
    pl.col("rel_ent").str.starts_with("sz:ds_open-ownership")
)

df_er_oo.head(3)

ent,rel_ent,sem_rel,why,evidence
str,str,str,str,str
"""sz:123""","""sz:ds_open-ownership_155526128…","""skos:exactMatch""","""INITIAL""","""INITIAL"""
"""sz:121""","""sz:ds_open-ownership_152340900…","""skos:exactMatch""","""INITIAL""","""INITIAL"""
"""sz:54""","""sz:ds_open-ownership_112192523…","""skos:exactMatch""","""INITIAL""","""INITIAL"""


In [29]:
conn.execute("DROP TABLE IF EXISTS Matched");
conn.execute(
    """
    CREATE REL TABLE IF NOT EXISTS Matched (
        FROM Entity TO OpenSanctions,
        FROM Entity TO OpenOwnership,
        sem_rel STRING,
        why STRING,
        evidence STRING
    )
    """
);
conn.execute("COPY Matched FROM df_er_os (from='Entity', to='OpenSanctions')");
conn.execute("COPY Matched FROM df_er_oo (from='Entity', to='OpenOwnership')");

Finally, close the database connection.

---

In [30]:
db.close()

---